In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder     .appName("TP Spark - NYC Taxi")     .getOrCreate()

In [3]:
df = spark.read     .option("header", True)     .option("inferSchema", True)     .csv("/home/jovyan/train.csv")

In [19]:
#Nombre de lignes
df.count()

1458644

In [20]:
df_filtered = df.filter(
    col("passenger_count") > 2
)

In [26]:
#Afficher les passager superieur à 2
df_filtered.count()

214726

In [27]:
#Afficher les 5 premières lignes
df.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|id3858529|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|    

In [28]:
#Afficher le schéma
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)



In [29]:
#Liste des colonnes
print(df.columns)

['id', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag', 'trip_duration']


In [30]:
#Nombre de colonnes
print("Nombre de colonnes :", len(df.columns))

Nombre de colonnes : 11


In [31]:
#Statistiques descriptives
df.describe().show()

+-------+---------+-------------------+------------------+-------------------+--------------------+-------------------+-------------------+------------------+------------------+
|summary|       id|          vendor_id|   passenger_count|   pickup_longitude|     pickup_latitude|  dropoff_longitude|   dropoff_latitude|store_and_fwd_flag|     trip_duration|
+-------+---------+-------------------+------------------+-------------------+--------------------+-------------------+-------------------+------------------+------------------+
|  count|  1458644|            1458644|           1458644|            1458644|             1458644|            1458644|            1458644|           1458644|           1458644|
|   mean|     NULL| 1.5349502688798637|1.6645295219395548| -73.97348630489282|  40.750920908391734|  -73.9734159469458|   40.7517995149002|              NULL| 959.4922729603659|
| stddev|     NULL|0.49877715390740074| 1.314242167823115|0.07090185842270368|0.032881186257633095|0.070643268

In [32]:
# Etape 1 convertir la durée en minute
df = df.withColumn(
    "trip_duration_in_minutes",
    col("trip_duration") / 60
)

In [34]:
# Etape 2 renommé les colonnes
df = df.withColumnRenamed("pickup_datetime", "pickup_time")
df = df.withColumnRenamed("dropoff_datetime", "dropoff_time")

In [35]:
# Etape 3 supprimer store_and_fwd_flag
df = df.drop("store_and_fwd_flag")

In [36]:
#afficher le schéma
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- trip_duration_in_minutes: double (nullable = true)



In [37]:
#afficher les noms des colonnes
print(df.columns)

['id', 'vendor_id', 'pickup_time', 'dropoff_time', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'trip_duration', 'trip_duration_in_minutes']


In [38]:
#afficher quelques lignes (5)
df.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+-------------+------------------------+
|       id|vendor_id|        pickup_time|       dropoff_time|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|trip_duration|trip_duration_in_minutes|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+-------------+------------------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|          455|       7.583333333333333|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|          663|                   11.05|
|id3858529|        2|2016-01-19 11:

In [40]:
#vérifier la nouvelle colonne sur 5lignes
df.select("trip_duration", "trip_duration_in_minutes").show(5)

+-------------+------------------------+
|trip_duration|trip_duration_in_minutes|
+-------------+------------------------+
|          455|       7.583333333333333|
|          663|                   11.05|
|         2124|                    35.4|
|          429|                    7.15|
|          435|                    7.25|
+-------------+------------------------+
only showing top 5 rows



In [41]:
#vérifier que store_and_fwd_flag a été supprimée
print("store_and_fwd_flag" in df.columns)

False


In [42]:
# Étape 1 — Filtrage
df_filtered = df.filter(
    (col("passenger_count") > 2) &
    (col("trip_duration") < 600)
)

In [43]:
#Afficher quelques résultats (5)
df_filtered.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+-----------------+------------------+------------------+-------------+------------------------+
|       id|vendor_id|        pickup_time|       dropoff_time|passenger_count|  pickup_longitude|  pickup_latitude| dropoff_longitude|  dropoff_latitude|trip_duration|trip_duration_in_minutes|
+---------+---------+-------------------+-------------------+---------------+------------------+-----------------+------------------+------------------+-------------+------------------------+
|id0801584|        2|2016-01-30 22:01:40|2016-01-30 22:09:03|              6|-73.98285675048828|40.74219512939453|-73.99208068847656|40.749183654785156|          443|       7.383333333333334|
|id1813257|        1|2016-06-17 22:34:59|2016-06-17 22:40:40|              4| -73.9690170288086|40.75783920288086|-73.95740509033203| 40.76589584350586|          341|       5.683333333333334|
|id1870624|        1|2016-01-05 15:29:54

In [45]:
#nombre de lignes obtenues
df_filtered.count()

93362

In [46]:
#Étape 2 — Conversion du type
df_filtered = df_filtered.withColumn(
    "vendor_id",
    col("vendor_id").cast("string")
)

In [47]:
#Vérifier le schéma
df_filtered.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- trip_duration_in_minutes: double (nullable = true)



In [48]:
#Étape 3 — Tri
df_filtered = df_filtered.orderBy(
    col("trip_duration_in_minutes").desc()
)

In [49]:
#afficher le filtre 
df_filtered.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+-----------------+------------------+------------------+-------------+------------------------+
|       id|vendor_id|        pickup_time|       dropoff_time|passenger_count|  pickup_longitude|  pickup_latitude| dropoff_longitude|  dropoff_latitude|trip_duration|trip_duration_in_minutes|
+---------+---------+-------------------+-------------------+---------------+------------------+-----------------+------------------+------------------+-------------+------------------------+
|id1505196|        1|2016-02-19 23:21:13|2016-02-19 23:31:12|              3|-73.98793029785156| 40.7650146484375|-73.98309326171875| 40.74917984008789|          599|       9.983333333333333|
|id3966722|        2|2016-01-07 07:49:41|2016-01-07 07:59:40|              5|-73.98091888427734|40.73395919799805|-73.98287963867188| 40.74795913696289|          599|       9.983333333333333|
|id1514772|        2|2016-06-08 12:56:41

In [51]:
#Vérifier la structure avant union
print("Colonnes df :", df.columns)
print("Colonnes df_filtered :", df_filtered.columns)

Colonnes df : ['id', 'vendor_id', 'pickup_time', 'dropoff_time', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'trip_duration', 'trip_duration_in_minutes']
Colonnes df_filtered : ['id', 'vendor_id', 'pickup_time', 'dropoff_time', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'trip_duration', 'trip_duration_in_minutes']


In [54]:
#Étape 4 — Union
df_union = df.union(df_filtered)

In [56]:
#afficher union
df_union.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+-------------+------------------------+
|       id|vendor_id|        pickup_time|       dropoff_time|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|trip_duration|trip_duration_in_minutes|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+-------------+------------------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|          455|       7.583333333333333|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|          663|                   11.05|
|id3858529|        2|2016-01-19 11:

In [57]:
# Combien de trajets correspondent aux conditions ?
print("Nombre de trajets :", df_filtered.count())

Nombre de trajets : 93362


In [58]:
#Quel est le trajet le plus long parmi les résultats filtrés ?
df_filtered.select(
    "trip_duration",
    "trip_duration_in_minutes"
).orderBy(
    col("trip_duration").desc()
).show(1)

+-------------+------------------------+
|trip_duration|trip_duration_in_minutes|
+-------------+------------------------+
|          599|       9.983333333333333|
+-------------+------------------------+
only showing top 1 row



In [59]:
#Quel est le type de vendor_id avant et après cast() ?
print("Type de vendor_id avant :", df.schema["vendor_id"].dataType)
print("Type de vendor_id après :", df_filtered.schema["vendor_id"].dataType)

Type de vendor_id avant : IntegerType()
Type de vendor_id après : StringType()


In [61]:
#Que se passe-t-il sur le nombre de lignes après union() ?
print("Avant union :", df.count())
print("Après union :", df_union.count())

Avant union : 1458644
Après union : 1552006


In [62]:
#Est-ce que union() supprime automatiquement les doublons ?
#non

In [ ]:
#Exercice 4 : 

In [63]:
from pyspark.sql.functions import col, to_date, avg, count
import time

In [64]:
#Étape 1 — Durée moyenne par jour
# Créer la colonne trip_date
df = df.withColumn("trip_date", to_date(col("pickup_time")))

In [65]:
#durée moyenne des trajets pour chaque jour ordre cronologique
df_daily = df.groupBy("trip_date").agg(
    avg("trip_duration_in_minutes").alias("avg_duration_minutes")
).orderBy("trip_date")

In [66]:
#afficher daily
df_daily.show()

+----------+--------------------+
| trip_date|avg_duration_minutes|
+----------+--------------------+
|2016-01-01|  15.344666294331187|
|2016-01-02|   14.00141277641278|
|2016-01-03|  15.411118106931113|
|2016-01-04|  14.185311028500621|
|2016-01-05|   24.25574680732926|
|2016-01-06|  13.974971713057258|
|2016-01-07|  13.868928400226604|
|2016-01-08|  14.533914610928829|
|2016-01-09|  13.474910624077088|
|2016-01-10|  12.484319381092927|
|2016-01-11|  14.416233736485248|
|2016-01-12|  14.217120006842865|
|2016-01-13|   14.29094696969697|
|2016-01-14|  15.087211550078532|
|2016-01-15|   15.72388452980443|
|2016-01-16|  14.828039862074187|
|2016-01-17|   14.74427617961391|
|2016-01-18|  11.927129396398922|
|2016-01-19|   15.94629863036719|
|2016-01-20|  15.076423879604231|
+----------+--------------------+
only showing top 20 rows



In [67]:
#Étape 2 — Repartitionnement
#A. Sans repartitionnement
start = time.time()

In [68]:
df_daily_no_repartition = df.groupBy("trip_date").agg(
    avg("trip_duration_in_minutes").alias("avg_duration_minutes")
)

In [69]:
df_daily_no_repartition.show()

+----------+--------------------+
| trip_date|avg_duration_minutes|
+----------+--------------------+
|2016-03-01|   14.96489348131139|
|2016-04-25|   14.14257256687536|
|2016-05-03|  17.233981711402784|
|2016-01-28|   17.05130589304902|
|2016-06-02|  18.085713713713712|
|2016-02-04|  15.703515578369345|
|2016-05-26|  17.471980051717765|
|2016-01-13|   14.29094696969697|
|2016-04-22|  17.963929546018356|
|2016-06-16|  17.210094803370783|
|2016-01-01|  15.344666294331187|
|2016-01-19|   15.94629863036719|
|2016-05-09|  17.105852151287056|
|2016-02-08|  13.562395686826074|
|2016-05-23|  14.394089041095883|
|2016-02-03|  16.484789644012945|
|2016-03-25|  16.518841533979803|
|2016-06-17|  17.004370243064077|
|2016-04-30|  15.186034548215652|
|2016-05-27|  15.943895528995137|
+----------+--------------------+
only showing top 20 rows



In [70]:
time_no_repartition = time.time() - start

In [94]:
print("Partitions :", df_daily_no_repartition.rdd.getNumPartitions())
print("Temps sans repartitionnement :", time_no_repartition, "secondes")

Partitions : 1
Temps sans repartitionnement : 31.56774878501892 secondes


In [80]:
#B. Avec repartition()
df_repartition = df.repartition("trip_date")

In [81]:
df_daily_repartition = df_repartition.groupBy("trip_date").agg(
    avg("trip_duration_in_minutes").alias("avg_duration_minutes")
)

In [82]:
df_daily_repartition.show()

+----------+--------------------+
| trip_date|avg_duration_minutes|
+----------+--------------------+
|2016-03-01|  14.964893481311403|
|2016-04-25|  14.142572566875355|
|2016-05-03|   17.23398171140274|
|2016-01-28|  17.051305893048923|
|2016-06-02|  18.085713713713687|
|2016-02-04|  15.703515578369325|
|2016-05-26|  17.471980051717708|
|2016-01-13|  14.290946969697009|
|2016-04-22|  17.963929546018363|
|2016-06-16|  17.210094803370737|
|2016-01-01|  15.344666294331223|
|2016-01-19|   15.94629863036716|
|2016-05-09|  17.105852151287028|
|2016-02-08|  13.562395686826061|
|2016-05-23|  14.394089041095917|
|2016-02-03|  16.484789644012928|
|2016-03-25|  16.518841533979817|
|2016-06-17|  17.004370243064034|
|2016-04-30|  15.186034548215611|
|2016-05-27|  15.943895528995109|
+----------+--------------------+
only showing top 20 rows



In [83]:
time_repartition = time.time() - start

In [84]:
print("Partitions après repartition :", df_repartition.rdd.getNumPartitions())
print("Temps avec repartition :", time_repartition, "secondes")

Partitions après repartition : 9
Temps avec repartition : 153.89913606643677 secondes


In [85]:
#C. Avec coalesce()
df_coalesce = df.coalesce(2)

In [86]:
df_daily_coalesce = df_coalesce.groupBy("trip_date").agg(
    avg("trip_duration_in_minutes").alias("avg_duration_minutes")
)

In [87]:
df_daily_coalesce.show()

+----------+--------------------+
| trip_date|avg_duration_minutes|
+----------+--------------------+
|2016-03-01|   14.96489348131139|
|2016-04-25|  14.142572566875339|
|2016-05-03|   17.23398171140279|
|2016-01-28|  17.051305893048966|
|2016-06-02|  18.085713713713726|
|2016-02-04|  15.703515578369345|
|2016-05-26|   17.47198005171778|
|2016-01-13|  14.290946969696979|
|2016-04-22|  17.963929546018374|
|2016-06-16|    17.2100948033708|
|2016-01-01|  15.344666294331208|
|2016-01-19|   15.94629863036719|
|2016-05-09|  17.105852151287095|
|2016-02-08|  13.562395686826054|
|2016-05-23|  14.394089041095874|
|2016-02-03|   16.48478964401296|
|2016-03-25|    16.5188415339798|
|2016-06-17|  17.004370243064077|
|2016-04-30|  15.186034548215606|
|2016-05-27|  15.943895528995172|
+----------+--------------------+
only showing top 20 rows



In [88]:
time_coalesce = time.time() - start

In [89]:
print("Partitions après coalesce :", df_coalesce.rdd.getNumPartitions())
print("Temps avec coalesce :", time_coalesce, "secondes")

Partitions après coalesce : 2
Temps avec coalesce : 231.1208462715149 secondes


In [96]:
print("Sans repartitionnement :",df_daily_no_repartition.rdd.getNumPartitions(),"partition", time_no_repartition, "s")
print("Avec repartition()     :",df_daily_repartition.rdd.getNumPartitions(),"partition", time_repartition, "s")
print("Avec coalesce()        :",df_coalesce.rdd.getNumPartitions(),"partition", time_coalesce, "s")

Sans repartitionnement : 1 partition 31.56774878501892 s
Avec repartition()     : 8 partition 153.89913606643677 s
Avec coalesce()        : 2 partition 231.1208462715149 s


In [99]:
#QUESTION 
#Quelle méthode est la plus rapide dans votre environnement ?
#La methode la plus rapide est sans partition 
#Le repartition() améliore-t-il systématiquement les performances ?
#NON
#Quel est l'impact du shuffle ?
#il rajoute du temps car les données sont redistribué entre les partition
#Quelle différence observez-vous entre repartition() et coalesce() ?
#repartition() peut augmenter ou réduire les partitions et provoque un shuffle, alors que coalesce() réduit les partitions avec généralement moins de shuffle

In [100]:
#Étape 3 — Mise en cache
# Mettre en cache
df_daily.cache()

DataFrame[trip_date: date, avg_duration_minutes: double]

In [101]:
# Premier accès
start = time.time()
df_daily.count()
time_first = time.time() - start

In [102]:
# Deuxième accès
start = time.time()
df_daily.count()
time_second = time.time() - start

In [103]:
# Troisième accès
start = time.time()
df_daily.count()
time_third = time.time() - start

In [104]:
print("Premier accès :", time_first, "s")
print("Deuxième accès :", time_second, "s")
print("Troisième accès :", time_third, "s")

Premier accès : 2.622398614883423 s
Deuxième accès : 0.08632183074951172 s
Troisième accès : 0.12836861610412598 s


In [105]:
# Vérifier le niveau de stockage
print("Niveau de stockage :", df_daily.storageLevel)

Niveau de stockage : Disk Memory Deserialized 1x Replicated


In [106]:
#Question 
#Le deuxième accès est-il plus rapide ?
#oui
#pourquoi ?
#parce que les données sont déjà en cache ducoup spark n’a pas besoin de les recalculer
#Que se passe-t-il lorsque les données sont mises en cache ?
#spark garde les données en mémoire pour pouvoir les reutiliser plus rapidement
#Dans quels cas le cache est-il intéressant ?
#quand on utilise plusieur fois le meme dataframe
#Pourquoi ne faut-il pas mettre systématiquement tous les DataFrames en cache ?
#parceque le cache utilise de la memoire et ca l'a remplis inutilement

In [113]:
#Étape 4 — Agrégation par jour et fournisseur
df_by_vendor = df.groupBy(
    "trip_date",
    "vendor_id"
).agg(
    count("*").alias("total_trips")
).orderBy(
    "trip_date",
    "vendor_id"
)

In [114]:
df_by_vendor.show(50)

+----------+---------+-----------+
| trip_date|vendor_id|total_trips|
+----------+---------+-----------+
|2016-01-01|        1|       3068|
|2016-01-01|        2|       4094|
|2016-01-02|        1|       2905|
|2016-01-02|        2|       3607|
|2016-01-03|        1|       2916|
|2016-01-03|        2|       3437|
|2016-01-04|        1|       3178|
|2016-01-04|        2|       3547|
|2016-01-05|        1|       3322|
|2016-01-05|        2|       3882|
|2016-01-06|        1|       3420|
|2016-01-06|        2|       3945|
|2016-01-07|        1|       3621|
|2016-01-07|        2|       4028|
|2016-01-08|        1|       3843|
|2016-01-08|        2|       4386|
|2016-01-09|        1|       3935|
|2016-01-09|        2|       4643|
|2016-01-10|        1|       3399|
|2016-01-10|        2|       4055|
|2016-01-11|        1|       3396|
|2016-01-11|        2|       3880|
|2016-01-12|        1|       3558|
|2016-01-12|        2|       4236|
|2016-01-13|        1|       3954|
|2016-01-13|        

In [115]:
#Question 
#Comment les trajets sont-ils répartis entre les fournisseurs ?
#Les trajets sont répartis entre les différent vendor_id avec un nb de trajet different
#Quelle opération Spark avez-vous utilisée ?
#groupBy() avec count() pour compter les trajets par fournisseur

In [116]:
#ÉTAPE 5 — Analyse du plan d'exécution avec Catalyst
#afficher le plan detailler
df_daily.explain(True)

== Parsed Logical Plan ==
'Sort ['trip_date ASC NULLS FIRST], true
+- Aggregate [trip_date#2937], [trip_date#2937, avg(trip_duration_in_minutes#2240) AS avg_duration_minutes#2963]
   +- Project [id#17, vendor_id#18, pickup_time#2253, dropoff_time#2266, passenger_count#21, pickup_longitude#22, pickup_latitude#23, dropoff_longitude#24, dropoff_latitude#25, trip_duration#27, trip_duration_in_minutes#2240, to_date(pickup_time#2253, None, Some(Etc/UTC), false) AS trip_date#2937]
      +- Project [id#17, vendor_id#18, pickup_time#2253, dropoff_time#2266, passenger_count#21, pickup_longitude#22, pickup_latitude#23, dropoff_longitude#24, dropoff_latitude#25, trip_duration#27, trip_duration_in_minutes#2240]
         +- Project [id#17, vendor_id#18, pickup_time#2253, dropoff_datetime#20 AS dropoff_time#2266, passenger_count#21, pickup_longitude#22, pickup_latitude#23, dropoff_longitude#24, dropoff_latitude#25, store_and_fwd_flag#26, trip_duration#27, trip_duration_in_minutes#2240]
            +-

In [119]:
#Question 
#Quelles transformations apparaissent dans le plan ?
# filter, withColumn, groupBy, agg, orderBy
#Quelles opérations sont visibles dans le plan physique ?
#Scan, HashAggregate, Sort, Exchange
#Pouvez-vous identifier une opération Exchange ?
#oui lorsque on utilise un shuffle
#Que signifie cette opération dans un traitement distribué ?
#ça signifie que les données sont redistribuées entre les partitions
#Le plan change-t-il après un repartition() ?
#le plan contient un Exchange supplémentaire
#Le comportement du plan change-t-il après une mise en cache ?
#on peut utiliser InMemoryTableScan
#Quel est le rôle de Catalyst ?
#optimiser le plan d'exécution


In [ ]:
#ÉTAPE 6 — Observer Spark UI

#Combien de Jobs avez-vous déclenchés ?
#75
#Combien de Stages sont associés à vos traitements ?
#111
#Quelles actions ont déclenché des Jobs ?
#show(), count(), collect() etc.
#Pouvez-vous identifier les opérations nécessitant une redistribution des données ?
#groupBy() et repartition()
#
